# Beacon v3 held-out evaluation
Tests the published continued adapter on all 233 English test conversations. No training. Private synthetic test data bundled; no credentials included.


In [ ]:
import base64, io, os, sys, subprocess, zipfile
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES']='0'
WORK=Path('/kaggle/working/beacon_eval'); WORK.mkdir(exist_ok=True)
BUNDLE='...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as z:
    assert all((WORK/p.filename).resolve().is_relative_to(WORK.resolve()) for p in z.infolist())
    z.extractall(WORK)
os.chdir(WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.17.0','peft==0.21.0','bitsandbytes==0.50.2','accelerate==1.15.0','pydantic==2.13.5'],check=True)
subprocess.run([sys.executable,'slm/evaluate_v3_release.py','--batch-size','16'],check=True)
result=Path('/kaggle/working/beacon_v3_evaluation.zip')
with zipfile.ZipFile(result,'w',zipfile.ZIP_DEFLATED) as z:
    for p in Path('outputs/beacon-training-v3/evaluation').glob('*'):
        if p.is_file(): z.write(p,p.name)
print('EVALUATION COMPLETE:', result)
